# M01 — StateGraph 基礎

本 notebook 對應同資料夾的 `README.md`，逐格執行即可。

我們會把 M00 的心智模型（State / node / edge）第一次寫成真的程式：
定義一個 State、寫兩個線性串接的 node、接上 START/END、compile、invoke，
最後把圖印成 ASCII 看結構。

## 1. 環境準備

載入共用 helper，讓範例與供應商無關。
本模組不一定要呼叫模型（圖的骨架純結構即可），但我們仍取得 `model`，
之後的練習可以選擇讓 node 真的去呼叫 LLM。

In [ ]:
# Load shared helpers so every notebook stays provider-agnostic.
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parents[1] / "_shared"))
from course_utils import get_model, load_env

load_env()
model = get_model()

## 2. 定義 State：圖的共享記憶體

State 用 `TypedDict` 列出整張圖會用到的欄位。
它是全圖共享的：每個 node 都讀同一份、寫同一份。
這裡設計成一張「逐步被填滿的表單」——一開始只有 `topic`，
之後由節點依序填上 `outline` 和 `draft`。

In [ ]:
from typing_extensions import TypedDict


class State(TypedDict):
    topic: str      # input: what to write about
    outline: str    # filled by generate_outline
    draft: str      # filled by write_draft

## 3. 寫 node：吃 state、回傳「只含要更新欄位」的 dict

每個 node 是一個普通函式：參數是 `state`，回傳一個 `dict`。
重點：回傳的 dict **只放這一步要更新的鍵**，沒提到的欄位 LangGraph 會原封不動保留。

- `generate_outline`：讀 `topic`，產生 `outline`。
- `write_draft`：讀 `topic` 與 `outline`，產生 `draft`。

這裡用字串拼接模擬產出（不呼叫 LLM），方便聚焦在「圖怎麼運作」。

In [ ]:
def generate_outline(state: State) -> dict:
    # read only what this node needs from the shared state
    topic = state["topic"]
    outline = f"《{topic}》三段式大綱：1) 是什麼 2) 為什麼重要 3) 怎麼開始"
    # return ONLY the field this node updates; topic/draft stay untouched
    return {"outline": outline}


def write_draft(state: State) -> dict:
    # this node can see fields filled by earlier nodes
    topic = state["topic"]
    outline = state["outline"]
    draft = f"草稿（主題：{topic}）\n依大綱展開 -> {outline}"
    return {"draft": draft}

## 4. 組圖：builder → add_node → add_edge → compile

流程固定五步：
1. `StateGraph(State)` 開一張綁定該 State 的空白圖（這是「藍圖」builder）。
2. `add_node` 把函式註冊成有名字的節點。
3. `add_edge` 宣告執行順序，並用 `START`/`END` 標出入口與出口。
4. `compile()` 把藍圖固化成可執行的 `graph`，順便做合法性檢查。

這裡接成線性鏈：`START → generate_outline → write_draft → END`。

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(State)
builder.add_node("generate_outline", generate_outline)
builder.add_node("write_draft", write_draft)

builder.add_edge(START, "generate_outline")
builder.add_edge("generate_outline", "write_draft")
builder.add_edge("write_draft", END)

graph = builder.compile()

## 5. 執行：invoke 後觀察 state 如何一步步被填入

給一個初始 state（只有 `topic` 有值），`invoke` 回傳的是**整份最終 state**（一個 dict）。
注意 `outline` 與 `draft` 是空的也沒關係——它們會被節點依序填上。

In [ ]:
result = graph.invoke({"topic": "向量資料庫", "outline": "", "draft": ""})

print("topic   :", result["topic"])
print("outline :", result["outline"])
print("draft   :", result["draft"])
# Expected output (大致):
# topic   : 向量資料庫
# outline : 《向量資料庫》三段式大綱：1) 是什麼 2) 為什麼重要 3) 怎麼開始
# draft   : 草稿（主題：向量資料庫）
#           依大綱展開 -> 《向量資料庫》三段式大綱：...

## 6. 視覺化：把圖結構印成 ASCII

`get_graph().draw_ascii()` 會印出文字版流程圖。
學圖的拓樸時，這是最快確認「邊有沒有接對」的方法。

In [ ]:
print(graph.get_graph().draw_ascii())
# Expected output (大致):
#       +-----------+
#       | __start__ |
#       +-----------+
#             *
#       +------------------+
#       | generate_outline |
#       +------------------+
#             *
#       +-------------+
#       | write_draft |
#       +-------------+
#             *
#       +---------+
#       | __end__ |
#       +---------+

## 🧪 練習 1：插入一個 node

在 `write_draft` 之後、`END` 之前，加一個 `polish` 節點：
讀 `draft`、回傳更新後的 `draft`（例如在結尾加上「（已潤稿）」字樣）。

提示：
1. 在 State 不用新增欄位（沿用 `draft` 即可）。
2. `builder.add_node("polish", polish)`。
3. 把邊改成 `write_draft → polish → END`（記得移除原本 `write_draft → END` 那條，
   重新建一個 builder 比較乾淨）。
4. 重新 `compile`、`invoke`，並再次 `draw_ascii()` 確認多了一格。

In [ ]:
# Your turn: define polish(state) -> dict, rewire the graph, then invoke.
# def polish(state: State) -> dict:
#     return {"draft": state["draft"] + "\n（已潤稿）"}

## 🧪 練習 2：讓某個 node 真的呼叫 LLM

把 `write_draft` 改成用第 1 格取得的 `model` 真正生成草稿：
用 `model.invoke(...)` 餵入 `topic` 與 `outline`，取 `.content` 當 `draft`。

提示：
- 範例：`resp = model.invoke(f"依大綱寫一段短文：{state['outline']}")`
- 然後 `return {"draft": resp.content}`。
- 這示範了 node 內部可以做任何事（呼叫模型、查資料庫…），只要最後回傳「要更新的欄位」dict。

In [ ]:
# Your turn: rewrite write_draft to call the model, then re-run the graph.

## 小結 & 下一步

你已經完成 LangGraph 的最小可運作骨架：
- 用 `TypedDict` 定義共享 State。
- 寫成函式的 node，回傳「只含要更新欄位」的 dict。
- 用 `START`/`END` 與 `add_edge` 接成線性圖。
- 走完 `compile → invoke`，並用 `draw_ascii()` 檢查結構。

目前一個欄位被更新時是「直接覆蓋」。下一個模組 **M02 — 狀態與 Reducer**
會教你用 `Annotated` + reducer（如 `add`、`add_messages`）做「累加」而非覆蓋，
讓多個節點能安全地往同一欄位追加資料（例如把對話訊息一則則疊進 `messages`）。